# Times Indonesia Debug

Notebook ini untuk validasi list berita dan sample artikel Times Indonesia sebelum scraper dimasukkan ke pipeline final.

In [14]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'scrapers_debug':
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scrapers.common import cutoff_date, parse_date
import importlib
from scrapers import timesindonesia as scraper
scraper = importlib.reload(scraper)

cutoff = cutoff_date()
print('ROOT:', ROOT)
print('cutoff:', cutoff)

ROOT: /Users/tsar/Library/Mobile Documents/com~apple~CloudDocs/main/college/pkl/project
cutoff: 2026-03-02 10:57:44.155492


In [15]:
MAX_PAGES = 200
TITLE_LIMIT = 90
SAMPLE_SIZE = None  # None = semua artikel dari source_df


def short_title(value, limit=TITLE_LIMIT):
    text = '' if pd.isna(value) else str(value).strip()
    return text if len(text) <= limit else text[: limit - 3] + '...'


def ensure_debug_columns(df):
    df = df.copy()
    if 'page_num' not in df.columns:
        df['page_num'] = pd.NA
    if 'published_date' not in df.columns:
        df['published_date'] = None
    df['published_dt'] = df['published_date'].apply(parse_date)
    return df


def print_debug_rows(df):
    for _, row in df.iterrows():
        page = row.get('page_num')
        page_text = '---' if pd.isna(page) else f'{int(page):03d}'
        date_text = row.get('published_date')
        date_text = 'None' if pd.isna(date_text) else str(date_text)
        rel_text = row.get('relative_date')
        date_source = row.get('date_source')
        if pd.isna(rel_text) or not str(rel_text).strip():
            rel_text = '' if pd.isna(date_source) else f' | src={date_source}'
        else:
            rel_text = f' | rel={rel_text}'
        print(f"page={page_text} | date={date_text}{rel_text} | title={short_title(row.get('title'))}")


def audit_list(df):
    df = ensure_debug_columns(df)
    print('total rows:', len(df))
    if len(df) == 0:
        print('No rows found.')
        return df

    print('first page:', df['page_num'].dropna().min() if df['page_num'].notna().any() else None)
    print('last page:', df['page_num'].dropna().max() if df['page_num'].notna().any() else None)
    print('newest date:', df['published_dt'].max())
    print('oldest date:', df['published_dt'].min())
    print('cutoff date:', cutoff)
    print('reached cutoff:', bool((df['published_dt'].dropna() < cutoff).any()))
    print('null parsed dates:', int(df['published_dt'].isna().sum()))
    print('unique urls:', df['url'].nunique() if 'url' in df.columns else None)

    print('\nrows per source category:')
    if 'source_category' in df.columns:
        display(df.groupby('source_category', dropna=False).size().reset_index(name='count'))

    print('\nrows per month:')
    display(
        df.assign(month=df['published_dt'].dt.to_period('M').astype(str))
        .groupby('month', dropna=False)
        .size()
        .reset_index(name='count')
    )

    print('\nrows per page:')
    display(
        df.groupby('page_num', dropna=False)
        .agg(rows=('url', 'count'), newest=('published_dt', 'max'), oldest=('published_dt', 'min'))
        .reset_index()
        .tail(60)
    )
    return df

## Live list scrape

Run cell ini untuk cek pagination live. Jika page 2 kosong, berarti search page Times Indonesia kemungkinan tidak expose pagination query `page=`.

In [16]:
list_df = scraper.scrape_list_auto(cutoff, max_pages=MAX_PAGES, include_older=True)
print_debug_rows(list_df)
list_df = audit_list(list_df)
list_output = ROOT / 'csv' / 'times_indonesia_list_debug.csv'
list_df.to_csv(list_output, index=False)
print('saved:', list_output)

Times Indonesia API GET offset=0: https://timesindonesia.co.id/api/news/all?news_type=search&offset=0&title=kabupaten+malang&limit=9
Times Indonesia API OK offset=0: rows=9
Times Indonesia API page=001 offset=0 raw_new=9 new=9 total=9 newest=2026-07-01 oldest=2026-06-26
Times Indonesia list page=001 date=2026-07-01 title=DTPHP Kabupaten Malang: Belum Ada Bukti Liburnya MBG Sebabkan Harga Sayur Turun
Times Indonesia list page=001 date=2026-07-01 title=Harga Sayur di Kabupaten Malang Turun Drastis, Ada Kaitan dengan Libur Program MBG?
Times Indonesia list page=001 date=2026-06-30 title=OJK: Aset Perbankan di Kabupaten Malang Tembus Rp36,3 Triliun
Times Indonesia list page=001 date=2026-06-30 title=6.300 Anak di Kabupaten Malang Belum Pernah Mengenyam Pendidikan
Times Indonesia list page=001 date=2026-06-30 title=Bupati: Masih Ada 31 ribu Anak Tidak Sekolah di Kabupaten Malang
Times Indonesia list page=001 date=2026-06-29 title=DPRD Kabupaten Malang Dorong Penyelesaian Sengketa Lahan Kali

,source_category,count
0,Ekonomi,35
1,Hukum dan Kriminal,4
2,Indonesia Positif,25
3,Kesehatan,2
4,Kopi TIMES,3
5,Kuliner,2
6,Liputan Khusus,2
7,Olahraga,11
8,Pemerintahan,42
9,Pendidikan,25



rows per month:


,month,count
0,2026-02,5
1,2026-03,53
2,2026-04,68
3,2026-05,77
4,2026-06,65
5,2026-07,2



rows per page:


,page_num,rows,newest,oldest
0,1,9,2026-07-01,2026-06-26
1,2,9,2026-06-26,2026-06-23
2,3,9,2026-06-22,2026-06-17
3,4,9,2026-06-17,2026-06-14
4,5,9,2026-06-13,2026-06-10
5,6,9,2026-06-10,2026-06-06
6,7,9,2026-06-05,2026-06-02
7,8,9,2026-06-01,2026-05-30
8,9,9,2026-05-29,2026-05-25
9,10,9,2026-05-25,2026-05-23


saved: /Users/tsar/Library/Mobile Documents/com~apple~CloudDocs/main/college/pkl/project/csv/times_indonesia_list_debug.csv


## Article sample extraction

Cell ini mengambil beberapa artikel dari `list_df` untuk cek kebersihan content.

In [17]:
article_rows = []

source_df = list_df if 'list_df' in globals() and len(list_df) else local_df
if SAMPLE_SIZE is not None:
    source_df = source_df.head(SAMPLE_SIZE)
print('article sample source rows:', len(source_df))

if len(source_df) == 0:
    print('Tidak ada row list untuk sample artikel. Jalankan cell local HTML atau live list dulu.')

for index, row in source_df.iterrows():
    try:
        article = scraper.extract_article(row)
        content = article.get('content') or ''
        article['content_len'] = len(content)
        article['has_literal_backslash_n'] = '\\n' in content
        article['has_actual_newline'] = '\n' in content
        article['has_double_blank_line'] = '\n\n' in content
        article['has_carriage_return'] = '\r' in content
        article_rows.append(article)
        print(f"article [{len(article_rows)}] len={len(content)} | date={article.get('published_date')} | title={short_title(article.get('title'))}")
        print((content[:300] + '...') if len(content) > 300 else content)
        print('-' * 80)
    except Exception as error:
        print(f"failed [{index + 1}]: {row.get('url')} | {error}")

article_columns = [
    'published_date',
    'content_len',
    'has_literal_backslash_n',
    'has_actual_newline',
    'has_double_blank_line',
    'title',
    'url',
]
article_df = pd.DataFrame(article_rows)
article_output = ROOT / 'csv' / 'times_indonesia_article_debug.csv'
article_df.to_csv(article_output, index=False)
print('saved:', article_output)

if article_df.empty:
    print('No article samples extracted. Cek error di atas; biasanya live detail URL gagal dibuka atau selector content belum cocok.')
else:
    display(article_df[[column for column in article_columns if column in article_df.columns]])


article sample source rows: 270
HTTP GET start: https://timesindonesia.co.id/ekonomi/597079/dtphp-kabupaten-malang-belum-ada-bukti-liburnya-mbg-sebabkan-harga-sayur-turun
HTTP GET ok: https://timesindonesia.co.id/ekonomi/597079/dtphp-kabupaten-malang-belum-ada-bukti-liburnya-mbg-sebabkan-harga-sayur-turun | status=200 | bytes=112253
article [1] len=3870 | date=2026-07-01 | title=DTPHP Kabupaten Malang: Belum Ada Bukti Liburnya MBG Sebabkan Harga Sayur Turun
MALANG – Penurunan harga sejumlah komoditas sayuran di Kabupaten Malang dalam dua pekan terakhir memunculkan pertanyaan di tengah masyarakat. Salah satu dugaan yang berkembang ialah menurunnya permintaan akibat Program Makan Bergizi Gratis (MBG ) yang dihentikan sementara selama masa libur sekolah. ...
--------------------------------------------------------------------------------
HTTP GET start: https://timesindonesia.co.id/ekonomi/597037/harga-sayur-di-kabupaten-malang-turun-drastis-ada-kaitan-dengan-libur-program-mbg
HTTP GET ok

,published_date,content_len,has_literal_backslash_n,has_actual_newline,has_double_blank_line,title,url
0,2026-07-01,3870,False,False,False,DTPHP Kabupaten Malang: Belum Ada Bukti Liburn...,https://timesindonesia.co.id/ekonomi/597079/dt...
1,2026-07-01,3576,False,False,False,"Harga Sayur di Kabupaten Malang Turun Drastis,...",https://timesindonesia.co.id/ekonomi/597037/ha...
2,2026-06-30,2605,False,False,False,OJK: Aset Perbankan di Kabupaten Malang Tembus...,https://timesindonesia.co.id/ekonomi/596952/oj...
3,2026-06-30,2568,False,False,False,6.300 Anak di Kabupaten Malang Belum Pernah Me...,https://timesindonesia.co.id/pendidikan/596949...
4,2026-06-30,2392,False,False,False,Bupati: Masih Ada 31 ribu Anak Tidak Sekolah d...,https://timesindonesia.co.id/pendidikan/596947...
...,...,...,...,...,...,...,...
265,2026-02-28,4082,False,False,False,"Efisiensi Bikin Resah, Fraksi DPRD Soroti Bans...",https://timesindonesia.co.id/peristiwa-daerah/...
266,2026-02-28,3487,False,False,False,PU Bina Marga Kabupaten Malang Tindaklanjuti R...,https://timesindonesia.co.id/peristiwa-daerah/...
267,2026-02-27,2211,False,False,False,"Disdik Kabupaten Malang Gelar DOR 2026, Perkua...",https://timesindonesia.co.id/peristiwa-daerah/...
268,2026-02-27,4242,False,False,False,Rakor MBG di Kabupaten Malang Jadi Ajang Curha...,https://timesindonesia.co.id/peristiwa-daerah/...
